### Imports

In [ ]:
import rasterio
import rasterio.mask
import geopandas as gpd
from shapely.geometry import mapping
import matplotlib.pyplot as plt
from pathlib import Path
import rioxarray

In [ ]:
# 1. Define paths to your data

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Inputs")
jamaica_metric_grid_crs = "EPSG:3448"


aboveground_carbon_tif = base_path / "aboveground_biomass_carbon_2010.tif"
belowground_carbon_tif = base_path / "belowground_biomass_carbon_2010.tif"


jamaica_boundary_path = base_path / "Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)

In [ ]:
# Inspect the boundary’s coordinate reference system (CRS)
print("Jamaica CRS:", jamaica_boundary.crs)

jam_crs = jamaica_boundary.crs

In [ ]:
# 2. Open and reproject the Aboveground Raster
aboveground_carbon_tif  = rioxarray.open_rasterio(aboveground_carbon_tif)
print("Original Aboveground Raster CRS:", aboveground_carbon_tif.rio.crs)

In [ ]:
# Reproject to Jamaica boundary CRS
aboveground_carbon_tif_reproj = aboveground_carbon_tif.rio.reproject(jam_crs)
print("Reprojected Aboveground Raster CRS:", aboveground_carbon_tif_reproj.rio.crs)

In [ ]:
# Save the reprojected aboveground raster
above_reproj_out = base_path / "aboveground_carbon_jamaica_crs.tif"
above_ras_reproj.rio.to_raster(above_reproj_out)

In [ ]:
# 3. Load the carbon rasters
with rasterio.open(aboveground_carbon_tif) as src:
    aboveground_profile = src.profile
    aboveground_crs = src.crs
    print("Aboveground raster CRS:", aboveground_crs)

with rasterio.open(belowground_carbon_tif) as src:
    belowground_profile = src.profile
    belowground_crs = src.crs
    print("Belowground raster CRS:", belowground_crs)

In [ ]:
# 4. Define a helper function to reproject a raster to a target CRS
def reproject_raster(input_raster, output_raster, target_crs):
    """
    Reproject 'input_raster' to 'target_crs' and save to 'output_raster'.
    """
    with rasterio.open(input_raster) as src:
        transform, width, height = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds
        )
        
        # Copy the original profile but update with new transform, width, height, and CRS
        profile = src.profile.copy()
        profile.update({
            'crs': target_crs,
            'transform': transform,
            'width': width,
            'height': height
        })
        
        with rasterio.open(output_raster, 'w', **profile) as dst:
            for band_index in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, band_index),
                    destination=rasterio.band(dst, band_index),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=target_crs,
                    resampling=Resampling.nearest  # or Resampling.bilinear, etc.
                )


In [ ]:
# 5. Reproject aboveground and belowground rasters to match Jamaica's CRS
aboveground_jam_crs = 'aboveground_carbon_jamaica_crs.tif'
belowground_jam_crs = 'belowground_carbon_jamaica_crs.tif'

reproject_raster(aboveground_carbon_tif, aboveground_jam_crs, jam_crs)
reproject_raster(belowground_carbon_tif, belowground_jam_crs, jam_crs)